In [1]:
from neo.io import NixIO
import quantities as pq

filename = "results/spike_trains.nix"
with NixIO(filename, mode="rw") as io:
    block = io.read_block()
print(block.segments[0].spiketrains[0].times)
print(f"Number of segments: {len(block.segments)}")
print(f"Number of spike_trains: {len(block.segments[0].spiketrains)}")

[4.0312 4.0334 4.0361 4.063  4.065  4.0736 4.1042 4.1062 4.1082 4.1254
 4.1274 4.1436 4.1456 4.1833 4.1853 4.1978 4.1998 4.286  4.2883 4.2937
 4.2957 4.4057 4.4208 4.5214 4.55   4.569  4.5879 4.5899 4.612  4.6474
 4.6494 4.7305 4.7325 4.8357 4.883  4.885  4.9377 5.1062 5.2063 5.2083
 5.2103 5.6529] s
Number of segments: 1
Number of spike_trains: 512


In [6]:
print(block.segments[0].spiketrains[450].annotations.keys())


dict_keys(['nix_name', 'neuron_id', 'forward_connections', 'weights'])


In [ ]:
"""Okay currently I have a list of all spikes
I want to end up with sets of 3-item tuples which state which neurons are connected in the binding 
neuron format.
This should be done to a n ms precision.
two options:
    two stage cull
    one stage cull

two stage cull:
    get sets of all neurons whose synapses are arranged n1->n2, n2->n3, n1->n3
    check that t(n1->n2) + t(n2->n3) ~= t(n1->n3)
    if not, remove the set from the list
"""


In [ ]:
metadata = {}
for i, values in enumerate(block.segments[0].spiketrains):
    metadata[i] = values.annotations

triples = []
# Get triples
for initial_neuron in range(len(metadata)):
    second_neurons = metadata[initial_neuron]['forward_connections']
    for second_neuron in second_neurons:
        third_neurons = metadata[second_neuron]['forward_connections']
        for third_neuron in third_neurons:
            if third_neuron in metadata[initial_neuron]['forward_connections']:
                triples.append((initial_neuron, second_neuron, third_neuron))

# cull triples
sparse_triples = triples
alpha = 20 * pq.ms # Threshold for time difference
for triple in sparse_triples:
    t1 = block.segments[0].spiketrains[triple[0]].annotations['t_stop']
    t2 = block.segments[0].spiketrains[triple[1]].annotations['t_stop']
    t3 = block.segments[0].spiketrains[triple[2]].annotations['t_stop']
    if t1 + t2 - t3 > alpha:
        triples.remove(triple)